In [ ]:
import torch
import time

DEVICE = "mps"

# Switch precision to float16 to engage the GPU Matrix Engines
DTYPE = torch.float16 

# 1. Create a massive, compute-heavy square matrix multiplication
# Matrix dimension N = 8192 yields 2 * (N^3) floating-point operations per run.
N = 8192
A = torch.randn((N, N), device=DEVICE, dtype=DTYPE)
B = torch.randn((N, N), device=DEVICE, dtype=DTYPE)

# 2. Warm up the GPU and compile/initialize the MPS Graph pipeline
print("Warming up the hardware pipeline...")
for _ in range(5):
    C = torch.mm(A, B)
torch.mps.synchronize()

# 3. Benchmark execution loops
M = 100
print(f"Running {M} iterations of {N}x{N} Matrix Multiplication...")

torch.mps.synchronize()
start_time = time.time()

for _ in range(M):
    C = torch.mm(A, B)

torch.mps.synchronize()
end_time = time.time()

# 4. Math breakdown
total_time = end_time - start_time
avg_time_per_iter = total_time / M

# Standard GEMM FLOP count: 2 * N^3 (one multiply + one add per element pairing)
flops_per_matrix_mul = 2 * (N ** 3)
tflops = (flops_per_matrix_mul / avg_time_per_iter) / 1e12

print("\n-------------------------------------------")
print(f"Average execution time per iteration: {avg_time_per_iter:.6f} seconds")
print(f"Measured Compute Performance: {tflops:.2f} TFLOPS")
print("-------------------------------------------")

Warming up the hardware pipeline...
Running 100 iterations of 8192x8192 Matrix Multiplication...

-------------------------------------------
Average execution time per iteration: 0.087556 seconds
Measured Compute Performance: 12.56 TFLOPS
-------------------------------------------
